In [ ]:
# Validate image and mask pairing
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "notebooks" / "main.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start the kernel inside the TerraMind project.")
DATA_DIR = PROJECT_ROOT / "data/raw/SECOND_train_set"
folders = ("im1", "im2", "label1", "label2")
if not DATA_DIR.is_dir():
    raise FileNotFoundError("SECOND_train_set is missing from TerraMind/data/raw.")

file_names = {folder: {p.name for p in (DATA_DIR / folder).glob("*.png")} for folder in folders}
all_names = set.union(*file_names.values())
missing = {folder: len(all_names - names) for folder, names in file_names.items()}
if not all_names or any(missing.values()):
    raise ValueError(f"Expected complete image/mask pairs. Missing files by folder: {missing}")

sample_names = sorted(all_names)
print(f"Complete pairs: {len(sample_names):,}")

In [ ]:
# Encode SECOND semantic masks (RGB -> class ID)
# Reference: https://www.mdpi.com/2072-4292/15/16/4095 (Table 2)
import numpy as np

SECOND_CLASSES = (
    ("No change", (255, 255, 255)),
    ("Water", (0, 0, 255)),
    ("Non-vegetated ground", (128, 128, 128)),
    ("Low vegetation", (0, 128, 0)),
    ("Trees", (0, 255, 0)),
    ("Buildings", (128, 0, 0)),
    ("Playgrounds", (255, 0, 0)),
)
palette = np.array([rgb for _, rgb in SECOND_CLASSES], dtype=np.uint8)


def encode_mask(rgb_mask):
    """Convert a uint8 RGB mask (H, W, 3) to class indices (H, W)."""
    if rgb_mask.ndim != 3 or rgb_mask.shape[-1] != 3 or rgb_mask.dtype != np.uint8:
        raise ValueError("Expected a uint8 RGB mask with shape (H, W, 3).")

    class_ids = np.full(rgb_mask.shape[:2], -1, dtype=np.int64)
    for class_id, rgb in enumerate(palette):
        class_ids[np.all(rgb_mask == rgb, axis=-1)] = class_id

    unknown = class_ids == -1
    if unknown.any():
        first_rgb = rgb_mask[unknown][0].tolist()
        raise ValueError(f"Unexpected mask color {first_rgb} ({unknown.sum():,} unknown pixels).")
    return class_ids

In [ ]:
# Split original pairs before extracting patches
SEED = 42
shuffled_names = np.random.default_rng(SEED).permutation(sorted(sample_names)).tolist()
split_at = int(0.8 * len(shuffled_names))
train_names, val_names = shuffled_names[:split_at], shuffled_names[split_at:]

assert set(train_names).isdisjoint(val_names)
assert set(train_names) | set(val_names) == set(sample_names)
print(f"Train: {len(train_names):,} pairs | Validation: {len(val_names):,} pairs")

In [ ]:
# Load aligned 256 x 256 patches on demand
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader


class SecondPatches(Dataset):
    def __init__(self, root, pair_names):
        self.root = root
        self.pair_names = pair_names

    def __len__(self):
        return len(self.pair_names) * 4

    def __getitem__(self, index):
        pair_index, tile = divmod(index, 4)
        name = self.pair_names[pair_index]
        top, left = (tile // 2) * 256, (tile % 2) * 256
        box = (left, top, left + 256, top + 256)
        arrays = []
        for folder in ("im1", "im2", "label1", "label2"):
            with Image.open(self.root / folder / name) as image:
                if image.size != (512, 512):
                    raise ValueError(f"Expected a 512 x 512 image: {folder}/{name}")
                arrays.append(np.array(image.crop(box).convert("RGB")))

        images = [torch.from_numpy(a).permute(2, 0, 1).float().div(255) for a in arrays[:2]]
        masks = [torch.from_numpy(encode_mask(a)) for a in arrays[2:]]
        return (*images, *masks)


train_dataset = SecondPatches(DATA_DIR, train_names)
val_dataset = SecondPatches(DATA_DIR, val_names)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=4, num_workers=0)

images_t1, images_t2, masks_t1, masks_t2 = next(iter(train_loader))
for image in (images_t1, images_t2):
    assert image.shape == (4, 3, 256, 256)
    assert image.dtype == torch.float32 and 0 <= image.min() <= image.max() <= 1
for mask in (masks_t1, masks_t2):
    assert mask.shape == (4, 256, 256)
    assert mask.dtype == torch.int64 and 0 <= mask.min() <= mask.max() <= 6

print(f"Train patches: {len(train_dataset):,} | Validation patches: {len(val_dataset):,}")
print(f"Batch images: {tuple(images_t1.shape)} | Masks: {tuple(masks_t1.shape)}")

## Siamese ResNet18

A shared ResNet18 encoder extracts features from both timestamps. The prediction head combines each timestamp's features with their absolute difference. The training cell below runs one epoch.

In [ ]:
# Define the shared ResNet18 encoder and the weighted training setup
from collections import Counter
from time import perf_counter

from torch import nn
from torch.nn import functional as F
from torchvision.models import resnet18


class SiameseResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet18(weights=None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])
        self.head = nn.Sequential(
            nn.Conv2d(1024, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_classes, kernel_size=1),
        )

    def forward(self, image_t1, image_t2):
        features_t1 = self.encoder(image_t1)
        features_t2 = self.encoder(image_t2)
        difference = (features_t1 - features_t2).abs()
        return tuple(
            F.interpolate(
                self.head(torch.cat((features, difference), dim=1)),
                size=image_t1.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            for features in (features_t1, features_t2)
        )


def resnet_class_weights(root, pair_names):
    counts_by_color = Counter()
    for name in pair_names:
        for folder in ("label1", "label2"):
            with Image.open(root / folder / name) as mask:
                rgb = mask.convert("RGB")
                counts_by_color.update({
                    color: count for count, color in rgb.getcolors(rgb.width * rgb.height)
                })

    counts = torch.tensor([counts_by_color[tuple(rgb)] for rgb in palette])
    if (counts <= 0).any() or counts.sum().item() != sum(counts_by_color.values()):
        raise ValueError("Expected all seven SECOND classes and no unknown colors in train masks.")
    weights = counts.float().rsqrt()
    return weights / weights.mean()

In [ ]:
# Shared training and validation functions
def train_resnet_epoch(model, loader, optimizer, criterion, device):
    model.train()
    start = perf_counter()
    loss_sum = 0.0

    for step, batch in enumerate(loader, start=1):
        image1, image2, target1, target2 = (tensor.to(device) for tensor in batch)
        optimizer.zero_grad(set_to_none=True)
        logits1, logits2 = model(image1, image2)
        loss = (criterion(logits1, target1) + criterion(logits2, target2)) / 2
        loss.backward()
        optimizer.step()
        batch_loss = loss.item()
        loss_sum += batch_loss

        if step == 1 or step % 200 == 0 or step == len(loader):
            print(f"{type(model).__name__} train {step}/{len(loader)} | batch loss={batch_loss:.4f}")

    return loss_sum / len(loader), (perf_counter() - start) / 60


def validate_resnet(model, loader, device):
    model.eval()
    matched = predicted = actual = correct_pair = 0

    with torch.inference_mode():
        for batch in loader:
            image1, image2, target1, target2 = (tensor.to(device) for tensor in batch)
            logits1, logits2 = model(image1, image2)
            pred1, pred2 = logits1.argmax(dim=1), logits2.argmax(dim=1)

            gt_change = (target1 != 0) | (target2 != 0)
            pred_change = (pred1 != 0) | (pred2 != 0)
            matched += (gt_change & pred_change).sum().item()
            predicted += pred_change.sum().item()
            actual += gt_change.sum().item()
            correct_pair += (gt_change & (pred1 == target1) & (pred2 == target2)).sum().item()

    return {
        "actual_changed": actual,
        "predicted_changed": predicted,
        "precision": matched / max(predicted, 1),
        "recall": matched / max(actual, 1),
        "f1": 2 * matched / max(predicted + actual, 1),
        "pair_accuracy": correct_pair / max(actual, 1),
    }



In [ ]:
# Train Weighted Siamese ResNet18 for one epoch, validate, and save
torch.manual_seed(SEED)
resnet_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet_model = SiameseResNet18(num_classes=len(SECOND_CLASSES)).to(resnet_device)
resnet_optimizer = torch.optim.AdamW(resnet_model.parameters(), lr=1e-3)
resnet_weights = resnet_class_weights(DATA_DIR, train_names)
resnet_criterion = nn.CrossEntropyLoss(weight=resnet_weights.to(resnet_device))
train_loader.generator.manual_seed(SEED)

print(f"Training Weighted Siamese ResNet18 on {resnet_device}")
resnet_train_loss, resnet_train_minutes = train_resnet_epoch(
    resnet_model, train_loader, resnet_optimizer, resnet_criterion, resnet_device
)
resnet_metrics = validate_resnet(resnet_model, val_loader, resnet_device)

print(f"Mean batch loss={resnet_train_loss:.4f} | Training time={resnet_train_minutes:.1f} min")
print(f"Validation | Actual changed={resnet_metrics['actual_changed']:,} | "
      f"Pred changed={resnet_metrics['predicted_changed']:,}")
print(f"Change precision={resnet_metrics['precision']:.3f} | "
      f"Recall={resnet_metrics['recall']:.3f} | F1={resnet_metrics['f1']:.3f}")
print(f"Both classes correct on changed pixels={resnet_metrics['pair_accuracy']:.1%}")

resnet_checkpoint = PROJECT_ROOT / "outputs/checkpoints/resnet18_weighted_epoch1.pt"
resnet_checkpoint.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "epoch": 1,
    "model_state_dict": resnet_model.state_dict(),
    "optimizer_state_dict": resnet_optimizer.state_dict(),
    "class_weights": resnet_weights,
    "train_generator_state": train_loader.generator.get_state(),
    "validation": resnet_metrics,
    "train_minutes": resnet_train_minutes,
}, resnet_checkpoint)
print(f"Saved to {resnet_checkpoint}")

## Siamese Transformer

A shared Transformer encoder processes 16 × 16 image patches. The cell below checks that both timestamps produce seven class scores per pixel.

In [ ]:
# Siamese Transformer and one-batch forward check
class SiameseTransformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, 192, kernel_size=16, stride=16)
        self.position = nn.Parameter(torch.empty(1, 256, 192))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.blocks = nn.Sequential(*[
            nn.TransformerEncoderLayer(
                d_model=192, nhead=6, dim_feedforward=768,
                batch_first=True, norm_first=True, activation="gelu",
            )
            for _ in range(4)
        ], nn.LayerNorm(192))
        self.head = nn.Sequential(
            nn.Conv2d(384, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, num_classes, 1),
        )

    def encode(self, image):
        patches = self.patch_embed(image)
        tokens = patches.flatten(2).transpose(1, 2)
        tokens = self.blocks(tokens + self.position)
        return tokens.transpose(1, 2).reshape_as(patches)

    def forward(self, image1, image2):
        features1, features2 = self.encode(image1), self.encode(image2)
        difference = (features1 - features2).abs()
        return tuple(
            F.interpolate(
                self.head(torch.cat((features, difference), dim=1)),
                size=image1.shape[-2:], mode="bilinear", align_corners=False,
            )
            for features in (features1, features2)
        )


torch.manual_seed(SEED)
transformer_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer_model = SiameseTransformer(len(SECOND_CLASSES)).to(transformer_device)
transformer_model.eval()
with torch.inference_mode():
    first, second, target1, target2 = (
        tensor.to(transformer_device)
        for tensor in (images_t1, images_t2, masks_t1, masks_t2)
    )
    logits1, logits2 = transformer_model(first, second)
    expected = (first.size(0), len(SECOND_CLASSES), *first.shape[-2:])
    assert logits1.shape == logits2.shape == expected
    loss = (F.cross_entropy(logits1, target1) + F.cross_entropy(logits2, target2)) / 2
    assert torch.isfinite(loss)

print(f"{transformer_device} | T1 {tuple(logits1.shape)} | T2 {tuple(logits2.shape)} | loss={loss.item():.4f}")


In [ ]:
# Smoke test Comparison setup: run this and the next two cells from a fresh kernel.
import math
import statistics
import sys
from pathlib import Path

import torch

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/models/siamese.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the kernel inside the TerraMind project.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.datasets.second_dataset import PALETTE, make_val_loader
from src.models.siamese import SiameseHybrid, SiameseResNet18, SiameseTransformer

comparison_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
comparison_loader = make_val_loader(project_root / "data/raw/SECOND_train_set")
comparison_classes = len(PALETTE)
checkpoint_dir = project_root / "outputs/checkpoints"
comparison_runs = [
    ("ResNet18", SiameseResNet18, "resnet18_weighted_epoch3.pt"),
    ("Transformer", SiameseTransformer, "transformer_weighted_epoch3.pt"),
    ("Hybrid", SiameseHybrid, "hybrid_weighted_epoch3.pt"),
]

def load_comparison_model(model_class, filename):
    checkpoint = torch.load(
        checkpoint_dir / filename, map_location="cpu", weights_only=True
    )
    if checkpoint["epoch"] != 3:
        raise ValueError(f"Expected an Epoch 3 checkpoint: {filename}")
    model = model_class(num_classes=comparison_classes)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model.eval()

print(f"Validation: {len(comparison_loader.dataset):,} patches | {comparison_device}")

In [ ]:
# Evaluate Epoch 3 checkpoints on the shared validation split.
@torch.inference_mode()
def evaluate_scd(model, loader, device):
    n = comparison_classes
    hist = torch.zeros((n, n), dtype=torch.int64)
    pair_tp = pair_predicted = pair_actual = 0
    for batch in loader:
        x1, x2, y1, y2 = (tensor.to(device) for tensor in batch)
        logits1, logits2 = model(x1, x2)
        p1, p2 = logits1.argmax(1), logits2.argmax(1)

        for target, pred in ((y1, p1), (y2, p2)):
            indices = (n * target + pred).flatten().cpu()
            hist += torch.bincount(indices, minlength=n**2).reshape(n, n)

        actual = (y1 != 0) | (y2 != 0)
        predicted = (p1 != 0) | (p2 != 0)
        pair_tp += (actual & predicted).sum().item()
        pair_predicted += predicted.sum().item()
        pair_actual += actual.sum().item()

    actual_positive = hist[1:, :].sum().item()
    predicted_positive = hist[:, 1:].sum().item()
    semantic_tp = hist.diag()[1:].sum().item()
    change_tp = hist[1:, 1:].sum().item()
    change_iou = change_tp / max(actual_positive + predicted_positive - change_tp, 1)

    changed_hist = hist.double()
    changed_hist[0, 0] = 0
    total = changed_hist.sum().item()
    kappa = 0.0
    if total:
        observed = changed_hist.trace().item() / total
        expected = changed_hist.sum(1).dot(changed_hist.sum(0)).item() / total**2
        if expected < 1:
            kappa = (observed - expected) / (1 - expected)

    return {
        "binary_f1": 2 * pair_tp / max(pair_predicted + pair_actual, 1),
        "f1_scd": 2 * semantic_tp / max(actual_positive + predicted_positive, 1),
        "sek": kappa * math.exp(change_iou - 1),
    }


comparison_results = {}
for name, model_class, filename in comparison_runs:
    print(f"Evaluating {name}, epoch 3 ...", flush=True)
    model = load_comparison_model(model_class, filename).to(comparison_device)
    scores = evaluate_scd(model, comparison_loader, comparison_device)
    comparison_results[name] = scores
    print(
        f"{name:12s} | Binary F1={scores['binary_f1']:.4f} | "
        f"F1-SCD={scores['f1_scd']:.4f} | SeK={scores['sek']:.4f}"
    )
    del model

In [ ]:
# Benchmark batch-size-1 GPU forward latency and extra allocated memory.
if torch.cuda.is_available():
    benchmark_device = torch.device("cuda")
    warmup, repeats = 30, 100

    image1, image2, _, _ = next(iter(comparison_loader))
    image1 = image1[:1].to(benchmark_device)
    image2 = image2[:1].to(benchmark_device)

    benchmark_results = {}
    with torch.inference_mode():
        for name, model_class, filename in comparison_runs:
            model = load_comparison_model(model_class, filename)
            torch.cuda.synchronize()
            baseline_bytes = torch.cuda.memory_allocated()
            model = model.to(benchmark_device)
            parameters = sum(p.numel() for p in model.parameters())

            for _ in range(warmup):
                model(image1, image2)
            torch.cuda.synchronize()

            events = [
                (torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True))
                for _ in range(repeats)
            ]
            torch.cuda.reset_peak_memory_stats()
            for start, end in events:
                start.record()
                model(image1, image2)
                end.record()

            torch.cuda.synchronize()
            times = [start.elapsed_time(end) for start, end in events]
            median = statistics.median(times)
            p95 = statistics.quantiles(times, n=100, method="inclusive")[94]
            peak_mib = (torch.cuda.max_memory_allocated() - baseline_bytes) / 1024**2
            benchmark_results[name] = dict(
                parameters=parameters, median_ms=median, p95_ms=p95, peak_mib=peak_mib
            )
            print(
                f"{name:12s} | Params={parameters / 1e6:.2f}M | "
                f"Median={median:.2f} ms | P95={p95:.2f} ms | Peak extra={peak_mib:.1f} MiB"
            )
            del model
else:
    print("CUDA unavailable. GPU benchmark skipped.")

## Hybrid CNN–Transformer

Shared ResNet18 layers extract features for the Transformer encoder. Run the optional training cell below to train for three epochs and save a separate set of checkpoints.

In [ ]:
# Optional: train a new Hybrid run for 3 epochs.
# Run data preparation, the shared training functions, and Comparison setup first.
hybrid_dir = checkpoint_dir / "hybrid_rerun"
hybrid_dir.mkdir(exist_ok=True)
hybrid_paths = [hybrid_dir / f"hybrid_weighted_epoch{epoch}.pt" for epoch in (1, 2, 3)]
if any(path.exists() for path in hybrid_paths):
    raise FileExistsError(f"Choose a new hybrid_dir for another run: {hybrid_dir}")

cnn_checkpoint = torch.load(
    checkpoint_dir / "resnet18_weighted_epoch1.pt", map_location="cpu", weights_only=True
)
hybrid_weights = cnn_checkpoint["class_weights"]
del cnn_checkpoint

torch.manual_seed(SEED)
train_loader.generator.manual_seed(SEED)
hybrid_model = SiameseHybrid(num_classes=comparison_classes).to(comparison_device)
hybrid_optimizer = torch.optim.AdamW(hybrid_model.parameters(), lr=1e-3)
hybrid_criterion = nn.CrossEntropyLoss(weight=hybrid_weights.to(comparison_device))

for epoch, path in enumerate(hybrid_paths, start=1):
    print(f"Hybrid Epoch {epoch}", flush=True)
    loss, minutes = train_resnet_epoch(
        hybrid_model, train_loader, hybrid_optimizer, hybrid_criterion, comparison_device
    )
    metrics = validate_resnet(hybrid_model, comparison_loader, comparison_device)
    torch.save({
        "epoch": epoch,
        "model_state_dict": hybrid_model.state_dict(),
        "optimizer_state_dict": hybrid_optimizer.state_dict(),
        "class_weights": hybrid_weights,
        "train_generator_state": train_loader.generator.get_state(),
        "validation": metrics,
        "train_minutes": minutes,
    }, path)
    print(f"Train={minutes:.1f} min | Mean loss={loss:.4f} | "
          f"Change F1={metrics['f1']:.4f} | Both classes correct={metrics['pair_accuracy']:.1%}")
    print(f"Saved to {path}")
